# De-excitation blip checks

Compares the nominal blip variables to their `_deexadded` twins (simulated MARLEY
nuclear de-excitation photon blips injected during `create_df`, see
`src/deex_blip_injection.py`), after the WC generic neutrino selection.
The nominal-vs-`_deexadded` comparison is the no-de-excitation vs
GENIE+INCL+MARLEY systematic bracket.

Also checks the injection bookkeeping: match rates, match-level fallbacks, and
donor-usage uniformity (each MARLEY library event and each iso1g blip donor
should be sampled approximately uniformly).

No systematics are applied here (the weights/detvar dataframes for the current
production have not been processed yet); statistical uncertainties only.


In [ ]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt

import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.file_locations import intermediate_files_location
from src.plot_helpers import make_histogram_plot
from src.df_helpers import lazy_height

# File Loading

In [ ]:
print("loading all_df.parquet...")
all_df = pl.scan_parquet(f"{intermediate_files_location}/all_df.parquet")
print(f"num events in all_df: {lazy_height(all_df)}")

deex_cols = [c for c in all_df.collect_schema().names() if c.endswith("_deexadded")]
print(f"num _deexadded columns: {len(deex_cols)}")

# Injection bookkeeping

Per-filetype summary of the de-excitation injection. Expectations (from the
GENIE+INCL+MARLEY sample): ~72% of matched overlay events have >= 1 photon and
the mean is ~1.26 photons/event; match level 0 (exact CC/NC x mode x
protons x neutrons key) should dominate, and level -2 (eligible but unmatched)
should never occur. Data / EXT / the 1-gamma overlays are skipped by design
(level -1).

In [ ]:
bk = (
    all_df.select(["filetype", "deex_marley_donor_index", "deex_marley_match_level",
                   "deex_n_photons", "deex_n_photons_injected", "deex_n_blips_injected"])
    .collect()
)
assert (bk["deex_marley_match_level"] != -2).all(), "eligible-but-unmatched events found!"

summary = (
    bk.group_by("filetype")
    .agg(
        pl.len().alias("events"),
        (pl.col("deex_marley_donor_index") >= 0).mean().alias("frac_matched"),
        (pl.col("deex_n_photons") > 0).mean().alias("frac_ge1_photon"),
        pl.col("deex_n_photons").mean().alias("mean_photons"),
        pl.col("deex_n_blips_injected").mean().alias("mean_blips_injected"),
    )
    .sort("events", descending=True)
)
with pl.Config(tbl_rows=30, float_precision=3):
    print(summary)

matched = bk.filter(pl.col("deex_marley_donor_index") >= 0)
print("\nmatch levels among matched events (0 = exact key, 1-3 = fallbacks):")
print(matched["deex_marley_match_level"].value_counts().sort("deex_marley_match_level"))

# Donor-usage uniformity

Left: how many times each MARLEY library event was sampled (among sampled
donors). Right: how many times each iso1g blip donor was used. Both should look
roughly Poisson around their mean -- a long tail of heavily reused donors would
mean some matching bin is too sparse.

In [ ]:
iso_all = (
    all_df.select("deex_iso1g_donor_indices")
    .filter(pl.col("deex_iso1g_donor_indices") != "")
    .collect()["deex_iso1g_donor_indices"]
    .str.split(",").explode().cast(pl.Int64)
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (usage, label) in zip(axes, [
    (matched["deex_marley_donor_index"].value_counts()["count"].to_numpy(),
     "MARLEY library event"),
    (iso_all.value_counts()["count"].to_numpy(), "iso1g blip donor"),
]):
    mean_use = usage.mean()
    bins = np.arange(-0.5, max(usage.max() + 1.5, 10), 1)
    ax.hist(usage, bins=bins, histtype="step", color="#0072B2")
    ax.axvline(mean_use, color="#D55E00", linestyle=":",
               label=f"mean = {mean_use:.2f}")
    ax.set_xlabel(f"times each {label} was sampled")
    ax.set_ylabel("donors")
    ax.set_yscale("log")
    ax.legend()
    ax.set_title(f"{len(usage):,} distinct donors used")
plt.tight_layout()
plt.show()

# Configuration

`base_presel` mirrors `simple_generic_histogram.ipynb`: drop the raw 1-gamma
overlay samples and require a reconstructed neutrino energy (the WC generic
selection). Plots use the full runs 1-5 prediction weighting
(`wc_net_weight_full_pred`, no data role).

For each variable, three plots: the stacked nominal prediction, the stacked
`_deexadded` prediction, and a direct overlay of the two totals with their
ratio.

In [ ]:
SEL_TITLE = "WC Generic Selection"
weight_var = "wc_net_weight_full_pred"

base_presel = all_df.filter(
    ~pl.col("filetype").is_in(["isotropic_one_gamma_overlay", "delete_one_gamma_overlay", "fullosc_overlay"])
    & (pl.col("wc_kine_reco_Enu") > 0)
)


def plot_deex_pair(var, bins, display_var):
    """Stacked prediction for the nominal variable and its _deexadded twin."""
    pred = base_presel.filter(pl.col(weight_var).is_not_null())
    for suffix, tag in [("", "nominal"), ("_deexadded", "MARLEY de-excitation added")]:
        make_histogram_plot(
            pred_sel_df=pred, include_data=False, include_ratio=False,
            bins=bins, var=var + suffix, display_var=display_var,
            net_weight_var=weight_var,
            title=f"{SEL_TITLE} \u2014 {tag}",
        )


def plot_deex_overlay(var, bins, display_var):
    """Direct overlay of the two total predictions, with their ratio."""
    cols = (base_presel.filter(pl.col(weight_var).is_not_null())
            .select([var, var + "_deexadded", weight_var]).collect())
    w = cols[weight_var].to_numpy()
    h0, _ = np.histogram(cols[var].to_numpy(), bins=bins, weights=w)
    h1, _ = np.histogram(cols[var + "_deexadded"].to_numpy(), bins=bins, weights=w)

    fig, (ax, axr) = plt.subplots(2, 1, figsize=(7, 6), sharex=True,
                                  height_ratios=[3, 1])
    ax.stairs(h0, bins, color="black", label="nominal (no de-excitation)")
    ax.stairs(h1, bins, color="#D55E00", label="MARLEY de-excitation added")
    ax.set_yscale("log")
    ax.set_ylabel("Weighted events")
    ax.legend()
    ax.set_title(f"{SEL_TITLE} \u2014 {display_var}")
    ratio = np.divide(h1, h0, out=np.full_like(h1, np.nan), where=h0 > 0)
    axr.stairs(ratio, bins, color="#D55E00")
    axr.axhline(1, color="gray", linestyle=":")
    axr.set_ylabel("deex / nominal")
    axr.set_xlabel(display_var)
    plt.tight_layout()
    plt.show()


def plot_deex_variable(var, bins, display_var):
    plot_deex_pair(var, bins, display_var)
    plot_deex_overlay(var, bins, display_var)

# Blips near the WC reconstructed neutrino vertex (no quality cuts)

In [ ]:
plot_deex_variable("wc_blip_nWithin_75cm", np.arange(-0.5, 20.5, 1),
                   "Blips within 75 cm of WC reco $\\nu$ vertex")

In [ ]:
plot_deex_variable("wc_blip_minDist", np.linspace(0, 100, 26),
                   "Distance from WC reco $\\nu$ vertex to closest blip (cm)")

# Sphere / cone counting variables (with blip quality cuts)

These apply the sphere quality cuts (2+ planes, not touching a track, not by a
dead wire, > 15 cm from tracks) around the WC shower vertex, so the injected
blips are partly absorbed by the cuts -- the shift should be visibly smaller
than for the raw `nWithin` counts.

In [ ]:
plot_deex_variable("blip_sphere_n", np.arange(-0.5, 12.5, 1),
                   "Quality blips in 75 cm sphere around WC shower vertex")

In [ ]:
plot_deex_variable("blip_sphere_sumE", np.linspace(0, 25, 26),
                   "Summed quality-blip energy in 75 cm sphere (MeV)")

In [ ]:
plot_deex_variable("blip_no_shower_cone_n", np.arange(-0.5, 12.5, 1),
                   "Quality blips in sphere, outside shower cone")

In [ ]:
plot_deex_variable("blip_no_shower_cone_no_backtrack_cones_nonproton_n",
                   np.arange(-0.5, 10.5, 1),
                   "Non-proton quality blips, no shower/backtrack cones")

# Expectations

- Roughly 72% of matched overlay events receive >= 1 MARLEY photon (mean ~1.26),
  but after blip efficiency, TPC clipping, and out-of-TPC vertices, the mean
  injected reco blips per overlay event is ~0.4.
- `wc_blip_nWithin_75cm` should shift up by ~0.2 on average -- visible as a
  slight shift of the multiplicity distribution toward higher counts and a
  ratio above 1 that grows with blip count.
- The sphere/cone quality-cut variables shift less (their cuts absorb part of
  the injection), and proton-tagged variables should barely move (de-excitation
  blips are electron-like).
- This is the systematic variation itself: any selection variable that shifts
  appreciably here identifies where the de-excitation modeling uncertainty
  enters the analysis.